# py-tradeSeq — function-by-function R⇄Python parity

For users migrating R tradeSeq code to Python. Side-by-side calls per public function on the same fixture.

## 1. Setup

In [1]:
import os, sys, json, subprocess
for k in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'): os.environ[k]='8'
from pathlib import Path
import numpy as np, pandas as pd
NB = Path('.').resolve(); PORT = NB.parent if NB.name == 'examples' else NB
sys.path.insert(0, str(PORT)); sys.path.insert(0, str(PORT.parent/'omicverse-rebuildr'/'engine'))
import pytradeseq
from parity_metrics import compute_parity
# Use the same dumps from compare_R_vs_Python
ref = json.loads((PORT/'data'/'reference_output.json').read_text())
cand = json.loads((PORT/'data'/'candidate_output.json').read_text())
print(f"genes: {ref['n_genes']}, cells: {ref['n_cells']}, lineages: {ref['n_lineages']}")

genes: 100, cells: 140, lineages: 2


## 2. Function-by-function

### 2.1 `fitGAM`

> Fit a NB-GAM per gene per lineage.

| R name | Python | Type | Default | Description |
|---|---|---|---|---|
| `counts` | `counts` | matrix | — | genes × cells |
| `sds` | `sds` | dict (or `pseudotime`+`cellWeights` directly) | — | Slingshot output |
| `nknots` | `nknots` | int | `6` | spline basis size |
| `weights` | `weights` | matrix | NULL | optional gene × cell weights |
| `offset` | `offset` | vector | log(libsize) | per-cell offset |
| `family` | `family` | str | `"nb"` | "nb" or "poisson" |
| `parallel` | `parallel` | bool | FALSE/False | joblib parallelism |
| — | `seed` | int | `42` | **new in Python** — multinomial assignment RNG |

**R**:
```r
sce <- fitGAM(counts = counts, sds = sds, nknots = 4)
```

**Python**:
```python
gams = pytradeseq.fitGAM(counts, sds=sds, nknots=4)
```

### 2.2 `associationTest`

> Test whether expression depends on pseudotime (joint across lineages).

| R | Python | Default | Description |
|---|---|---|---|
| `models` | `gams` | — | from fitGAM |
| `contrastType` | `contrastType` | `"consecutive"` | (cosmetic; only "consecutive" implemented) |
| `lineages` | `lineages` | FALSE/False | per-lineage results vs pooled |

**Comparison vs R**:

In [2]:
from scipy.stats import spearmanr
r_p = np.array(ref['associationTest']['pvalue'])
p_p = np.array(cand['associationTest']['pvalue'])
mask = np.isfinite(r_p) & np.isfinite(p_p) & (r_p > 0) & (p_p > 0)
rho = spearmanr(-np.log10(r_p[mask]), -np.log10(p_p[mask]))[0]
top10_overlap = len(set(ref['associationTest']['top50'][:10]) & set(cand['associationTest']['top50'][:10])) / 10
print(f'Spearman on -log10(p): {rho:.4f}')
print(f'top-10 overlap:         {top10_overlap:.2f}')
print(f'  → {"PASS" if rho > 0.70 else "FAIL"}')

Spearman on -log10(p): 0.8385
top-10 overlap:         0.80
  → PASS


### 2.3 `startVsEndTest`

> Test whether expression differs between pseudotime endpoints.

| R | Python | Default | Description |
|---|---|---|---|
| `models` | `gams` | — | from fitGAM |
| `pseudotimeValues` | `pseudotimeValues` | NULL/None | explicit (start, end); defaults to min/max |

**Comparison vs R**:

In [3]:
r_p = np.array(ref['startVsEndTest']['pvalue'])
p_p = np.array(cand['startVsEndTest']['pvalue'])
mask = np.isfinite(r_p) & np.isfinite(p_p) & (r_p > 0) & (p_p > 0)
rho = spearmanr(-np.log10(r_p[mask]), -np.log10(p_p[mask]))[0]
print(f'Spearman: {rho:.4f}  (NOTE: known partial parity, see RECONSTRUCTION_REPORT.md §6)')

Spearman: 0.4489  (NOTE: known partial parity, see RECONSTRUCTION_REPORT.md §6)


### 2.4 `nknots` and `evaluateK`

| R | Python | Description |
|---|---|---|
| `nknots(sce)` | `pytradeseq.nknots(gams)` | return nknots used |
| `evaluateK(counts, sds, k=3:10)` | `pytradeseq.evaluateK(counts, sds=sds, k_range=range(3, 11))` | BIC scan |

## 3. Aggregate verdict

| Function | Metric | Threshold | Result | Pass |
|---|---|---|---|---|
| `fitGAM` | (no direct parity; consumed by tests) | — | — | — |
| `associationTest` | Spearman -log10 p | ≥ 0.70 | 0.84 (typical) | ✅ |
| `associationTest top50` | Jaccard | ≥ 0.60 | 0.67 | ✅ |
| `startVsEndTest` | Spearman -log10 p | — | 0.45 (approximate) | 🟡 |

See [`RECONSTRUCTION_REPORT.md`](../RECONSTRUCTION_REPORT.md) §6 for limitations + v0.2 roadmap.